In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, classification_report

# ==========================================
# 1. GENERATE REPRODUCIBLE DATASET
# ==========================================
np.random.seed(42)
n_samples = 2000

data = {
    'Tenure': np.random.randint(1, 72, n_samples),
    'MonthlyCharges': np.random.uniform(20, 120, n_samples),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.25, 0.25]),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'TechSupport': np.random.choice(['Yes', 'No'], n_samples, p=[0.3, 0.7]),
}

df = pd.DataFrame(data)

# Inject realistic dependencies to build predictable patterns
churn_prob = (
    (df['Contract'] == 'Month-to-month').astype(float) * 0.4 +
    (df['MonthlyCharges'] > 80).astype(float) * 0.2 -
    (df['Tenure'] / 72.0) * 0.4 +
    (df['TechSupport'] == 'No').astype(float) * 0.1
)
df['Churn'] = (churn_prob + np.random.normal(0, 0.1, n_samples) > 0.25).astype(int)

# ==========================================
# 2. DATA SPLITTING & PREPROCESSING
# ==========================================
X = df.drop('Churn', axis=1)
y = df['Churn']

# Stratify ensure train and test splits have equal proportions of churners
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ['Tenure', 'MonthlyCharges']
categorical_features = ['Contract', 'PaymentMethod', 'TechSupport']

# Use ColumnTransformer to strictly prevent data leakage during scaling
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        # sparse_output=False guarantees a clean NumPy array output
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

# Fit exclusively on training data; transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# ==========================================
# 3. TRAINING AND PERFORMANCE BENCHMARK
# ==========================================
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

print("="*60)
print("             CUSTOMER CHURN MODEL METRICS           ")
print("="*60)

for name, model in models.items():
    # Fit model
    model.fit(X_train_processed, y_train)

    # Predict discrete values and continuous probabilities
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    # Calculate target benchmarks
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    print(f"\n📌 {name}:")
    print(f"   • Accuracy : {acc:.4f}")
    print(f"   • Recall   : {rec:.4f}  <-- Critical Business Metric")
    print(f"   • ROC-AUC  : {auc:.4f}")


             CUSTOMER CHURN MODEL METRICS           

📌 Logistic Regression:
   • Accuracy : 0.8800
   • Recall   : 0.8105  <-- Critical Business Metric
   • ROC-AUC  : 0.9521

📌 Random Forest:
   • Accuracy : 0.8575
   • Recall   : 0.7843  <-- Critical Business Metric
   • ROC-AUC  : 0.9484

📌 XGBoost:
   • Accuracy : 0.8700
   • Recall   : 0.8170  <-- Critical Business Metric
   • ROC-AUC  : 0.9506
